In [4]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/mnt/wuji/ywx/psg-pretrain')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 预处理Pipeline:

1. 跑 wuji-preprocess: 从原始通道信号得到预处理后的信号，保存为npz文件 （可能的bug：duration为0，去修改/opt/miniconda3/envs/wj/lib/python3.11/site-packages/wuji_preprocess/runner.py的duration初始值为inf）

```wuji-preprocess /mnt/wuji/ywx/psg-pretrain/preprocess/hsp_8_channels.yaml -j 64```

2. 跑 wuji-index：划分数据集，得到基础index文件，包含指向预处理后的信号npz文件的路径，保存为csv文件

```wuji-index index/psg_pretrain/mesa.yaml```

3. 在这个notebook中将第二步得到的文件和hsp元数据合并，保存为完整csv文件

In [3]:
import pandas as pd
import re

# Step 1: 读取 path 索引文件和基本信息文件
df_path = pd.read_csv("index/hsp_path_indices.csv")       # 含 path,patient_id,split,duration
df_meta = pd.read_csv("index/hsp_basic_info.csv")         # 含 BDSPPatientID, SiteID, visit, basic info
df_extra = pd.read_csv("index/hsp_questionnaire.csv", low_memory=False)  # 含 问卷数据

# Step 2: 从 path 中提取 SiteID、BDSPPatientID、visit
def extract_info(path):
    match = re.search(r"sub-S(\d{13})/ses-(\d+)", path)
    if match:
        full_id = match.group(1)      # e.g. 0001111189075
        visit = int(match.group(2))   # e.g. 1 from ses-1
        site_id = "S" + full_id[:4]   # S0001
        bdsp_id = full_id[4:]         # 字符串 '111189075'
        return pd.Series([site_id, bdsp_id, visit])
    else:
        return pd.Series([None, None, None])

df_path[['SiteID', 'BDSPPatientID', 'visit']] = df_path['path'].apply(extract_info)

# Step 3: 字段类型转换保持一致
df_path['BDSPPatientID'] = df_path['BDSPPatientID'].astype(str)
df_meta['BDSPPatientID'] = df_meta['BDSPPatientID'].astype(str)
df_meta['visit'] = df_meta['visit'].astype(int)
df_extra['BDSPPatientID'] = df_extra['BDSPPatientID'].astype(str)
df_extra['visit'] = df_extra['visit'].astype(int)
df_path['visit'] = df_path['visit'].astype(int)

# Step 4: 合并 path + basic_info
merged = pd.merge(df_path, df_meta, on=['SiteID', 'BDSPPatientID', 'visit'], how='left')

# Step 5: 合并上一步结果 + questionnaire
final_df = pd.merge(merged, df_extra, on=['BDSPPatientID', 'visit'], how='left')

# Step 6: 保存全部结果（完整字段）
final_df.to_csv("index/hsp_psg_pretrain.csv", index=False)

# Step 7: 只保留感兴趣的字段，保存精简版
columns_to_keep = [
    "path", "AgeAtVisit", "SexDSC", "EthnicGroupDSC", "split", "duration",
    "dxAbnormalHeartRhythm", "dxAnxiety", "dxCancer"
]
filtered_df = final_df[[col for col in columns_to_keep if col in final_df.columns]]

# 重命名列名为更简洁的形式
column_rename_mapping = {
    "SexDSC": "sex",
    "EthnicGroupDSC": "ethnic_group",
    "AgeAtVisit": "age",
    "dxAbnormalHeartRhythm": "abnormal_heart_rhythm",
    "dxAnxiety": "anxiety",
    "dxCancer": "cancer"
}
filtered_df = filtered_df.rename(columns=column_rename_mapping)

filtered_df.to_csv("index/hsp_psg_pretrain.csv", index=False)

# Step 8: 显示筛选后的数据预览
print(filtered_df.head())


                                                path   age     sex  \
0  /DATA/disk1/ywx/wuji/hsp_8_channels/sub-S00011...  69.0  Female   
1  /DATA/disk1/ywx/wuji/hsp_8_channels/sub-S00011...  62.0    Male   
2  /DATA/disk1/ywx/wuji/hsp_8_channels/sub-S00011...  80.0  Female   
3  /DATA/disk1/ywx/wuji/hsp_8_channels/sub-S00011...  71.0    Male   
4  /DATA/disk1/ywx/wuji/hsp_8_channels/sub-S00011...  71.0    Male   

   ethnic_group  split  duration abnormal_heart_rhythm anxiety cancer  
0  Not Hispanic  train   28395.0                   NaN     NaN    NaN  
1  Not Hispanic  train   27648.0                     0       0      0  
2  Not Hispanic  train   28686.0                   NaN     NaN    NaN  
3  Not Hispanic  train   26096.0                     1       0      0  
4  Not Hispanic  train   28828.0                     1       0      0  


In [6]:
# import pandas as pd
# import re
# from pathlib import Path

# # ===== 路径配置 =====
# PRETRAIN_CSV = Path("index/hsp_psg_pretrain.csv")     # 原始精简版文件
# LABEL_CSV = Path("index/disease_index_full.csv")      # 新疾病标签文件
# OUT_CSV = Path("index/hsp_with_disease.csv")          # 输出文件路径

# # ===== 1) 读取 pretrain 文件 =====
# df = pd.read_csv(PRETRAIN_CSV)

# # 从 path 中提取 patient key：sub-Sxxxxxxxxxxxxx/ses-y
# def extract_pid_key(p):
#     m = re.search(r"(sub-S\d{13}/ses-\d+)", str(p))
#     return m.group(1) if m else None

# df["pid_key"] = df["path"].apply(extract_pid_key)

# # ===== 2) 读取新疾病标签，只保留 dataset == 'hsp' =====
# lab = pd.read_csv(LABEL_CSV, low_memory=False)
# lab = lab[lab["dataset"].astype(str).str.lower().eq("hsp")].copy()

# # 清洗 patient_id
# def normalize_patient_id(x):
#     s = str(x).strip()
#     s = s.lstrip("/")
#     s = s.split("?")[0].split("#")[0]
#     s = s.split(".")[0]
#     return s

# lab["patient_id"] = lab["patient_id"].apply(normalize_patient_id)

# # ===== 3) 获取所有疾病列（全集）并转为 Int64 =====
# disease_cols = [c for c in lab.columns if c not in ("dataset", "patient_id")]

# for c in disease_cols:
#     lab[c] = pd.to_numeric(lab[c], errors="coerce").round().astype("Int64")

# lab = lab[["patient_id"] + disease_cols].drop_duplicates("patient_id")

# # ===== 4) 删除 df 中旧的疾病列（不再保留旧标签）=====
# old_known = {"abnormal_heart_rhythm", "anxiety", "cancer"}
# drop_cols = [c for c in df.columns if c in old_known or c in disease_cols]
# if drop_cols:
#     df = df.drop(columns=drop_cols)

# # ===== 5) 合并 =====
# merged = df.merge(lab, how="left", left_on="pid_key", right_on="patient_id")
# merged = merged.drop(columns=["pid_key", "patient_id"])

# # ===== 6) 保存 =====
# merged.to_csv(OUT_CSV, index=False)

# # ===== 7) 输出统计信息 =====
# n_total = len(df)
# n_matched = merged[disease_cols].notna().any(axis=1).sum()

# print(f"[OK] 已保存到：{OUT_CSV}")
# print(f"[INFO] 总样本数：{n_total}")
# print(f"[INFO] 至少匹配到一个新疾病标签的样本数：{n_matched}")

# unmatched = merged[merged[disease_cols].isna().all(axis=1)]
# if len(unmatched) > 0:
#     print(f"[WARN] 有 {len(unmatched)} 个样本未匹配到任何疾病标签，示例：")
#     print(unmatched[['path']].head(10).to_string(index=False))

[OK] 已保存到：index/hsp_with_disease.csv
[INFO] 总样本数：24323
[INFO] 至少匹配到一个新疾病标签的样本数：8547
[WARN] 有 15776 个样本未匹配到任何疾病标签，示例：
                                                            path
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111189075/ses-1.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111189848/ses-1.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111190905/ses-1.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111190905/ses-2.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111190905/ses-3.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111190905/ses-4.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111191757/ses-1.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111192352/ses-1.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111192352/ses-2.npz
/DATA/disk1/ywx/wuji/hsp_9_channels/sub-S0001111192396/ses-1.npz


In [22]:
import pandas as pd
import re
from pathlib import Path
import numpy as np

# ===== 路径 =====
PRETRAIN_CSV = "index/hsp_psg_pretrain.csv"
LABEL_CSV    = "index/hsp_disease_fixed.csv"
OUT_CSV      = "index/hsp_with_disease_fixed.csv"

OUT_DIR      = Path("index")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LAB_COUNT_CSV         = OUT_DIR / "hsp_disease_counts_lab.csv"
LAB_POS_COUNT_CSV     = OUT_DIR / "hsp_disease_pos_counts_lab.csv"
MERGED_COUNT_CSV      = OUT_DIR / "hsp_disease_counts_merged.csv"
MERGED_POS_COUNT_CSV  = OUT_DIR / "hsp_disease_pos_counts_merged.csv"
COUNT_DIFF_CSV        = OUT_DIR / "hsp_disease_counts_diff.csv"

# ===== 小工具 =====
def clean_cols(cols):
    return [str(c).replace("\ufeff", "").strip() for c in cols]

def extract_key(s: str):
    m = re.search(r"(sub-S\d+/ses-\d+)", str(s))
    return m.group(1) if m else None

def base_name(name: str) -> str:
    return re.sub(r"\.\d+$", "", str(name))

def first_non_empty(series: pd.Series):
    for x in series:
        if pd.notna(x) and str(x) != "":
            return x
    return pd.NA

# ===== 1) 读取 =====
# 关键：禁止自动识别 NA，读入后再自定义清洗
lab = pd.read_csv(LABEL_CSV, low_memory=False, na_values=[], keep_default_na=False)
pre = pd.read_csv(PRETRAIN_CSV, low_memory=False)
lab.columns = clean_cols(lab.columns)
pre.columns = clean_cols(pre.columns)

# 去掉字符串首尾空格；空白字符串 -> NaN
for df_ in (lab,):
    for c in df_.columns:
        if df_[c].dtype == "object":
            df_[c] = df_[c].astype(str).str.strip()
            df_[c] = df_[c].replace({"": pd.NA})

if "path" not in pre.columns:
    raise ValueError("预训练表缺少 'path' 列")

# ===== 2) 构建匹配键 =====
pre["key"] = pre["path"].astype(str).apply(extract_key)

lab["key"] = None
if "session_id" in lab.columns:
    lab["key"] = lab["session_id"].astype(str).apply(extract_key)
if lab["key"].isna().any() and "path" in lab.columns:
    lab.loc[lab["key"].isna(), "key"] = lab["path"].astype(str).apply(extract_key)
lab = lab.dropna(subset=["key"]).copy()

# ===== 3) 识别疾病列（排除非疾病字段）=====
non_disease = {
    "path","session_id","key","patient_id","dataset","split","duration",
    "age","sex","ethnic_group"
}
disease_cols_lab = [c for c in lab.columns if c not in non_disease]

# ===== 3.1 将疾病列标准化为 0/1（保持“有标签”=非Na，“阳性”=1）=====
true_like  = {"1","y","yes","true","t"}
false_like = {"0","n","no","false","f"}

for c in disease_cols_lab:
    s = lab[c]
    if s.dtype == "object":
        s_lower = s.str.lower()
        s = np.where(s_lower.isin(true_like), 1,
            np.where(s_lower.isin(false_like), 0, s))
        lab[c] = s
    lab[c] = pd.to_numeric(lab[c], errors="coerce")

# ===== 3.2 标签表整体体检 =====
rows_total_lab = len(lab)
rows_with_any_lab = lab[disease_cols_lab].notna().any(axis=1).sum()
print(f"[CHK-LAB] 标签表行数: {rows_total_lab}")
print(f"[CHK-LAB] 标签表中至少一个疾病非空的行数: {rows_with_any_lab}")

# ===== 4) 合并前：合并 lab 的重复疾病列（.1/.2）=====
groups = {}
for c in disease_cols_lab:
    groups.setdefault(base_name(c), []).append(c)

for bname, cols in groups.items():
    if len(cols) > 1:
        lab[bname] = lab[cols].bfill(axis=1).iloc[:, 0]  # 行内取第一个非空
        lab.drop(columns=[c for c in cols if c != bname], inplace=True)

# 重新确认疾病列
disease_cols_lab = [c for c in lab.columns if c not in non_disease]

# 若同一 key 多行，逐列取第一个非空再聚合
if len(lab) != lab["key"].nunique() and disease_cols_lab:
    agg_map = {c: first_non_empty for c in disease_cols_lab}
    lab = lab.groupby("key", as_index=False).agg(agg_map)

# ===== 4.1 标签表各疾病“有标签/阳性”双口径计数 =====
lab_counts_labeled  = lab[disease_cols_lab].notna().sum().sort_values(ascending=False)
lab_counts_positive = (lab[disease_cols_lab] == 1).sum().sort_values(ascending=False)
print("\n[LAB] 各疾病“有标签(0或1)”样本数（前20）:")
print(lab_counts_labeled.head(20).to_string())
print("\n[LAB] 各疾病“阳性=1”样本数（前20）:")
print(lab_counts_positive.head(20).to_string())
lab_counts_labeled.to_csv(LAB_COUNT_CSV, header=["labeled_count"])
lab_counts_positive.to_csv(LAB_POS_COUNT_CSV, header=["positive_count"])
print(f"[OK] 标签表疾病计数已保存：{LAB_COUNT_CSV}, {LAB_POS_COUNT_CSV}")

# ===== ★ 方案A：合并前，先删除 pre 中与 lab 疾病列重名的列，避免右表列被加 _lab 后缀 =====
pre = pre.drop(columns=[c for c in disease_cols_lab if c in pre.columns], errors="ignore")

# ===== 5) 合并（左连接）=====
merged = pre.merge(lab[["key"] + disease_cols_lab], on="key", how="left", suffixes=("", "_lab"))

# ===== 6) 合并后：只用真实存在的疾病列 =====
disease_cols_in_merged = [c for c in disease_cols_lab if c in merged.columns]
missing = [c for c in disease_cols_lab if c not in merged.columns]
if missing:
    print("[WARN] 这些疾病列在合并结果中缺失，已跳过：", ", ".join(missing))

# ===== 7) 关键体检 =====
pre_keys = set(pre["key"].dropna().unique())
lab_keys = set(lab["key"].dropna().unique())
print("\n[CHK] pre uniq keys:", len(pre_keys))
print("[CHK] lab uniq keys:", len(lab_keys))
print("[CHK] key intersection:", len(pre_keys & lab_keys))

rows_with_any = merged[disease_cols_in_merged].notna().any(axis=1).sum() if disease_cols_in_merged else 0
print("[CHK] total rows:", len(merged))
print("[CHK] rows with >=1 disease (notna):", rows_with_any)

if disease_cols_in_merged:
    null_rate = merged[disease_cols_in_merged].isna().mean().sort_values(ascending=False)
    print("\n[CHK] top-10 columns NaN-rate:\n", null_rate.head(10))

if disease_cols_in_merged and rows_with_any < len(merged):
    unmatched = merged[merged[disease_cols_in_merged].isna().all(axis=1)]
    print("\n[CHK] unmatched examples (key,path):")
    print(unmatched[["key","path"]].head(10).to_string(index=False))

# ===== 8) 合并后：各疾病“有标签/阳性”双口径计数 =====
merged_counts_labeled  = merged[disease_cols_in_merged].notna().sum().sort_values(ascending=False)
merged_counts_positive = (merged[disease_cols_in_merged] == 1).sum().sort_values(ascending=False)
print("\n[MERGED] 各疾病“有标签(0或1)”样本数（前20）:")
print(merged_counts_labeled.head(20).to_string())
print("\n[MERGED] 各疾病“阳性=1”样本数（前20）:")
print(merged_counts_positive.head(20).to_string())
merged_counts_labeled.to_csv(MERGED_COUNT_CSV, header=["labeled_count"])
merged_counts_positive.to_csv(MERGED_POS_COUNT_CSV, header=["positive_count"])
print(f"[OK] 合并后疾病计数已保存：{MERGED_COUNT_CSV}, {MERGED_POS_COUNT_CSV}")

# ===== 9) 差异对比（按“有标签(0或1)”口径）=====
diff = pd.DataFrame({
    "lab_labeled": lab_counts_labeled,
    "merged_labeled": merged_counts_labeled.reindex(lab_counts_labeled.index).fillna(0).astype(int)
})
diff["diff(merged - lab)"] = diff["merged_labeled"] - diff["lab_labeled"]
print("\n[DIFF] 合并前后“有标签(0或1)”对比（前20）:")
print(diff.head(20).to_string())
diff.to_csv(COUNT_DIFF_CSV)
print(f"[OK] 计数差异已保存：{COUNT_DIFF_CSV}")

# ===== 10) 输出最终表（pretrain 原始列 + 疾病列）=====
pre_cols = [c for c in pre.columns if c != "key"]
out_df = merged[pre_cols + disease_cols_in_merged]

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(OUT_CSV, index=False)
print(f"\n[OK] 已保存到：{OUT_CSV}")


[CHK-LAB] 标签表行数: 11966
[CHK-LAB] 标签表中至少一个疾病非空的行数: 11966

[LAB] 各疾病“有标签(0或1)”样本数（前20）:
urologicorkidneyproblem                   11966
bipolardisorder                           11964
cerebrovasculardisease                    11964
seizure                                   11964
diabetes                                  11963
cancer                                    11963
posttraumaticstressdisorderptsd           11963
asthma                                    11962
headaches                                 11962
heartfailure                              11961
coronaryheartdisease                      11961
chronicpain                               11961
cognitiveimpairment                       11959
chronicobstructivepulmonarydiseasecopd    11959
anxiety                                   11958
depression                                11958
thyroiddisorder                           11958
hypertension                              11957
meningitis                                 8936
br

## MESA

wuji-index index/psg_pretrain/mesa.yaml

In [4]:
import pandas as pd
import numpy as np

# 可选：让 notebook 显示更全
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# ===== Step 1: 读取两个文件 =====
# 文件一：基本信息，示例列：age,sexM,ethnicity,race,dataset,patient_id,session_id
df_meta = pd.read_csv("index/mesa_metadata.csv")

# 文件二：路径与分割，示例列：path,patient_id,split,duration
# 注意这里的 patient_id 可能长成 "mesa-sleep-0001.npz"
df_path = pd.read_csv("index/mesa_path_indices.csv")

# ===== 从 path / patient_id 提取 session_id（去掉 .npz）=====
def to_session_id_from_any(s):
    if pd.isna(s): 
        return np.nan
    s = str(s).split("/")[-1]
    return s[:-4] if s.endswith(".npz") else s

df_path["session_id"] = df_path["path"].apply(to_session_id_from_any)
df_path["session_id"] = df_path["session_id"].fillna(
    df_path["patient_id"].apply(to_session_id_from_any)
)

# ===== 合并（以 df_path 为主）=====
merged = pd.merge(
    df_path, df_meta,
    on="session_id",
    how="left",
    validate="m:1"
)

# ===== 只保留一个 sex 列，值为 male / female =====
# 优先用 sexM（1=male, 0=female），如果没有，则尝试把已有的 sex 规范化
if "sexM" in merged.columns:
    sex_num = pd.to_numeric(merged["sexM"], errors="coerce")
    merged["sex"] = sex_num.map({1: "male", 0: "female"})
elif "sex" in merged.columns:
    s = merged["sex"].astype(str).str.strip().str.lower()
    merged["sex"] = np.where(s.isin(["m", "male", "1", "true"]), "male",
                      np.where(s.isin(["f", "female", "0", "false"]), "female", pd.NA))
else:
    merged["sex"] = pd.NA

# 丢弃 sexM（以及可能存在的旧 sex 列的副本）
cols_to_drop = [c for c in ["sexM"] if c in merged.columns]
merged = merged.drop(columns=cols_to_drop)

# ===== 保存（完整字段 + 精简字段）=====
merged.to_csv("index/mesa_merged_all.csv", index=False)

columns_to_keep = [
    "path", "session_id", "patient_id", "split", "duration",
    "age", "sex", "ethnicity", "race", "dataset"
]
out = merged[[c for c in columns_to_keep if c in merged.columns]].copy()
out.to_csv("index/mesa_psg_pretrain.csv", index=False)

print(out.head())

                                                       path       session_id  \
0  /DATA/disk1/ywx/wuji/mesa_8_channels/mesa-sleep-0001.npz  mesa-sleep-0001   
1  /DATA/disk1/ywx/wuji/mesa_8_channels/mesa-sleep-0002.npz  mesa-sleep-0002   
2  /DATA/disk1/ywx/wuji/mesa_8_channels/mesa-sleep-0006.npz  mesa-sleep-0006   
3  /DATA/disk1/ywx/wuji/mesa_8_channels/mesa-sleep-0012.npz  mesa-sleep-0012   
4  /DATA/disk1/ywx/wuji/mesa_8_channels/mesa-sleep-0014.npz  mesa-sleep-0014   

   split  duration  age     sex  ethnicity                       race dataset  
0  train   43199.0   70  female        NaN                      white    mesa  
1    val   39599.0   83  female        NaN                      white    mesa  
2  train   32399.0   57  female        NaN                   hispanic    mesa  
3    val   42830.0   80    male        NaN                      white    mesa  
4  train   50399.0   60  female        NaN  black or african american    mesa  


## MROS 

wuji-index index/psg_pretrain/mros.yaml

分 patient

In [ ]:
from wuji_index.builders.subject_splitter import SubjectSplitter
import pandas as pd

# 读取 csv
df = pd.read_csv("index/mros_path_indices.csv")
df_metadata = pd.read_csv("index/mros_metadata.csv")

# 原 patient_id 列里有 "mros-visit1-aa0001.npz"
df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)
df["patient_id"] = df["session_id"].str.extract(r"(aa\d+)", expand=False).str.upper()

s = SubjectSplitter(train=0.8, val=0.1, test=0.1)
df = s(df)

# 合并：按 patient_id 对齐
merged = df.merge(df_metadata, on="patient_id", how="left")

# ---- 列名映射 ----
columns_mapping = {
    "patient_id": "subject_id",
    "session_id": "session",
    "sexM": "sex",
    "bmi": "bmi",
    "age": "age",
    "ethnicity": "ethnicity",
    "race": "Race",
    "dataset": "Dataset",
    "path": "path",
    "duration": "duration",
    "split": "split"
}

# 重命名列（只改 mapping 里出现的）
merged = merged.rename(columns=columns_mapping)

# 保存结果
merged.to_csv("index/mros_psg_pretrain.csv", index=False)

In [2]:
import pandas as pd
import re
from pathlib import Path

# ===== 路径 =====
PATH_CSV = Path("index/mros_path_indices.csv")      # 只和 path 索引合并
LABEL_CSV = Path("index/disease_index_full.csv")
OUT_CSV = Path("index/mros_with_disease.csv")

# ===== 1) 读 mros_path_indices，并确保有 session_id =====
df = pd.read_csv(PATH_CSV)

# 如果没有 session_id，就从 path 提取：/xxx/mros-visit1-aa0001.npz -> mros-visit1-aa0001
if "session_id" not in df.columns:
    df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)

# 兼容一些上游表：有时会出现 session_id_x / session_id_y
key_candidates = [c for c in ["session_id", "session_id_x", "session_id_y"] if c in df.columns]
if not key_candidates:
    # 兜底再尝试从 path 提取一次
    df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)
    key_candidates = ["session_id"]

left_key = key_candidates[0]  # 优先使用 session_id，其次 _x/_y

# ===== 2) 读取新疾病标签，只保留 mros 并标准化 patient_id =====
lab = pd.read_csv(LABEL_CSV, low_memory=False)
lab = lab[lab["dataset"].astype(str).str.lower().eq("mros")].copy()

def normalize_pid(x: str) -> str:
    s = str(x).strip()
    s = s.lstrip("/")
    s = s.split("?")[0].split("#")[0]
    s = s.split(".")[0]   # 去扩展名
    return s

lab["patient_id"] = lab["patient_id"].apply(normalize_pid)

# 疾病列全集
disease_cols = [c for c in lab.columns if c not in ("dataset", "patient_id")]
for c in disease_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce").round().astype("Int64")

lab = lab[["patient_id"] + disease_cols].drop_duplicates("patient_id")

# ===== 3) 合并（左连接）：left_key <-> patient_id =====
merged = df.merge(lab, left_on=left_key, right_on="patient_id", how="left")

# 安全删除右表 ID（有就删，没有就忽略）
if "patient_id" in merged.columns:
    merged = merged.drop(columns=["patient_id"])

# （可选）把疾病列放到表末尾
base_cols = [c for c in merged.columns if c not in disease_cols]
merged = merged[base_cols + disease_cols]

# ===== 4) 保存 =====
merged.to_csv(OUT_CSV, index=False)

# ===== 5) 统计 =====
n_total = len(df)
n_hit = merged[disease_cols].notna().any(axis=1).sum()
print(f"[OK] 已保存：{OUT_CSV}")
print(f"[INFO] 总样本数：{n_total}；至少匹配到一个疾病标签的样本数：{n_hit}")

unmatched = merged[merged[disease_cols].isna().all(axis=1)]
if len(unmatched) > 0:
    print(f"[WARN] 有 {len(unmatched)} 个样本未匹配到疾病标签，示例：")
    show_cols = ["path"] + ([left_key] if left_key in merged.columns else [])
    print(unmatched[show_cols].head(10).to_string(index=False))

[OK] 已保存：index/mros_with_disease.csv
[INFO] 总样本数：3868；至少匹配到一个疾病标签的样本数：3868


## SHHS

In [ ]:
from wuji_index.builders.subject_splitter import SubjectSplitter
import pandas as pd

# 读取 csv
df = pd.read_csv("index/shhs_path_indices.csv")
df_metadata = pd.read_csv("index/shhs_metadata.csv")

# 原 patient_id 列里有 "mros-visit1-aa0001.npz"
df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)
df["patient_id"] = df["session_id"].str.extract(r"-(\d+)", expand=False)

s = SubjectSplitter(train=0.8, val=0.1, test=0.1)
df = s(df)

# 合并：按 patient_id 对齐
merged = df.merge(df_metadata, on="session_id", how="left")

# ---- 列名映射 ----
columns_mapping = {
    "patient_id": "subject_id",
    "session_id": "session",
    "sexM": "sex",
    "bmi": "bmi",
    "age": "age",
    "ethnicity": "ethnicity",
    "race": "Race",
    "dataset": "Dataset",
    "path": "path",
    "duration": "duration",
    "split": "split"
}

# 重命名列（只改 mapping 里出现的）
merged = merged.rename(columns=columns_mapping)

# 保存结果
merged.to_csv("index/shhs_psg_pretrain.csv", index=False)

In [2]:
import pandas as pd
from pathlib import Path

def merge_disease_labels(
    base_csv: str,
    diseases_dir: str,
    out_csv: str,
    disease_glob: str = "*_test.csv",
):
    """
    base_csv: 包含 [path, patient_id_x, split, duration, session_id, ...] 的主CSV
    diseases_dir: 存放疾病标签CSV的文件夹；文件名如 asthma_test.csv
    out_csv: 合并后的输出路径
    disease_glob: 匹配疾病CSV的通配符（默认 *_test.csv）
    """
    base = pd.read_csv(base_csv)

    # --- 保证主表里有 session_id; 若无就从 path 推导（取 basename 去掉扩展名）
    if "session_id" not in base.columns:
        if "path" not in base.columns:
            raise ValueError("主表既没有 session_id 也没有 path，无法对齐。")
        base = base.copy()
        base["session_id"] = base["path"].apply(lambda p: Path(str(p)).stem)

    # 统一 session_id 为字符串，避免 join 时类型不一致
    base["session_id"] = base["session_id"].astype(str)

    diseases_dir = Path(diseases_dir)
    disease_files = sorted(diseases_dir.glob(disease_glob))
    if not disease_files:
        raise FileNotFoundError(f"未在 {diseases_dir} 下找到 {disease_glob} 匹配的文件。")

    merged = base.copy()

    for f in disease_files:
        # 从文件名提取疾病名（去掉 *_test 后缀）
        stem = f.stem  # e.g., "asthma_test"
        disease_name = stem.replace("_test", "")  # -> "asthma"

        df = pd.read_csv(f)

        # 有些疾病CSV列名是 path,duration,split,withDisease
        # 我们从 path 里抽取 session_id（取 basename 去掉扩展名）
        if "path" not in df.columns:
            raise ValueError(f"{f} 缺少 'path' 列。")

        dfx = df.copy()
        dfx["session_id"] = dfx["path"].apply(lambda p: Path(str(p)).stem)
        dfx["session_id"] = dfx["session_id"].astype(str)

        # 取该疾病的标签列；若重复 session 出现（极少数情况），以“是否存在任意一次为阳性”为准（max）
        if "withDisease" not in dfx.columns:
            raise ValueError(f"{f} 缺少 'withDisease' 列。")

        label_series = (
            dfx[["session_id", "withDisease"]]
            .groupby("session_id", as_index=False)["withDisease"]
            .max()
            .rename(columns={"withDisease": disease_name})
        )

        # 左连接到主表；没有的自然变为 NaN
        merged = merged.merge(label_series, on="session_id", how="left")

        print(f"合并 {f.name} → 新增列: {disease_name}")

    # 保存
    merged.to_csv(out_csv, index=False)
    print(f"已保存到: {out_csv}")



BASE_CSV = "index/shhs_psg_pretrain.csv"             # 你的主CSV（含 patient/session 信息）
DISEASES_DIR = "index/shhs_d"                    # 存放 *test.csv 的文件夹
OUT_CSV = "index/shhs_d/shhs_d_merged_with_diseases.csv"

merge_disease_labels(BASE_CSV, DISEASES_DIR, OUT_CSV)


合并 allergiesorsinusproblems_test.csv → 新增列: allergiesorsinusproblems
合并 asthma_test.csv → 新增列: asthma
合并 bronchitis_test.csv → 新增列: bronchitis
合并 cerebrovasculardisease_test.csv → 新增列: cerebrovasculardisease
合并 chronicobstructivepulmonarydiseasecopd_test.csv → 新增列: chronicobstructivepulmonarydiseasecopd
合并 coronaryheartdisease_test.csv → 新增列: coronaryheartdisease
合并 diabetes_test.csv → 新增列: diabetes
合并 heartfailure_test.csv → 新增列: heartfailure
合并 hypertension_test.csv → 新增列: hypertension
合并 restlesslegsyndromerls_test.csv → 新增列: restlesslegsyndromerls
已保存到: index/shhs_d/shhs_d_merged_with_diseases.csv


In [7]:
import pandas as pd
import re
from pathlib import Path

# ===== 路径 =====
PATH_CSV = Path("index/mros_path_indices.csv")
LABEL_CSV = Path("index/disease_index_full.csv")
OUT_CSV = Path("index/mros_with_disease.csv")

# ===== 1) 读 mros_path_indices，并提取 session_id =====
df = pd.read_csv(PATH_CSV)
# 例如：/DATA/.../mros-visit1-aa0001.npz -> mros-visit1-aa0001
df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)

# ===== 2) 读取新疾病标签，只保留 dataset == 'mros'，标准化 patient_id =====
lab = pd.read_csv(LABEL_CSV, low_memory=False)
lab = lab[lab["dataset"].astype(str).str.lower().eq("mros")].copy()

def normalize_pid(x: str) -> str:
    s = str(x).strip()
    s = s.lstrip("/")
    s = s.split("?")[0].split("#")[0]
    s = s.split(".")[0]              # 去掉扩展名（若有）
    return s

lab["patient_id"] = lab["patient_id"].apply(normalize_pid)

# 疾病列全集（除 dataset、patient_id 外）
disease_cols = [c for c in lab.columns if c not in ("dataset", "patient_id")]

# 统一到可空整型 Int64
for c in disease_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce").round().astype("Int64")

lab = lab[["patient_id"] + disease_cols].drop_duplicates("patient_id")

# ===== 3) 合并（左连接）：session_id <-> patient_id =====
merged = df.merge(lab, left_on="session_id", right_on="patient_id", how="left")
merged = merged.drop(columns=["patient_id"])  # 去掉右表 ID

# （可选）把疾病列放到表末尾
base_cols = [c for c in merged.columns if c not in disease_cols]
merged = merged[base_cols + disease_cols]

# ===== 4) 保存 =====
merged.to_csv(OUT_CSV, index=False)

# ===== 5) 简要统计 =====
n_total = len(df)
n_hit = merged[disease_cols].notna().any(axis=1).sum()
print(f"[OK] 已保存：{OUT_CSV}")
print(f"[INFO] 样本总数：{n_total}；至少匹配到一个疾病标签的样本数：{n_hit}")

unmatched = merged[merged[disease_cols].isna().all(axis=1)]
if len(unmatched) > 0:
    print(f"[WARN] 有 {len(unmatched)} 个样本未匹配到疾病标签，示例：")
    print(unmatched[["path", "session_id"]].head(10).to_string(index=False))


KeyError: "['patient_id'] not found in axis"

### WSC

wuji-index index/psg_pretrain/wsc.yaml

In [3]:
from wuji_index.builders.subject_splitter import SubjectSplitter
import pandas as pd
import re

# 读取 csv
df = pd.read_csv("index/wsc_path_indices.csv")
df_meta = pd.read_csv("index/wsc_metadata.csv")

# 1) 从 path 提取 session_id：/..../wsc-visit1-10119-nsrr.npz -> wsc-visit1-10119-nsrr
df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)

# 2) 从 session_id 提取 patient_id：10119
#    形如 wsc-visit1-10119-nsrr
df["patient_id"] = df["session_id"].str.extract(r"-visit\d+-(\d+)-nsrr", expand=False)

# 3) 受试者分层划分（按 subject 粒度）
s = SubjectSplitter(train=0.8, val=0.1, test=0.1)
df = s(df)

# 4) 合并：按 session_id 对齐（避免同一 patient 多 visit 造成重复）
merged = df.merge(df_meta, on="session_id", how="left")

# ---- 列名映射（与 mros 模板保持一致）----
columns_mapping = {
    "patient_id": "subject_id",
    "session_id": "session",
    "sexM": "sex",
    "bmi": "bmi",
    "age": "age",
    "ethnicity": "ethnicity",
    "race": "Race",
    "dataset": "Dataset",
    "path": "path",
    "duration": "duration",
    "split": "split",
}

# 重命名列（只改 mapping 里出现的）
merged = merged.rename(columns=columns_mapping)

# 保存结果
merged.to_csv("index/wsc_psg_pretrain.csv", index=False)
print("[OK] 保存至 index/wsc_psg_pretrain.csv")


[OK] 保存至 index/wsc_psg_pretrain.csv


In [4]:
import pandas as pd
import re
from pathlib import Path

# ===== 路径 =====
PATH_CSV = Path("index/wsc_path_indices.csv")
LABEL_CSV = Path("index/disease_index_full.csv")
OUT_CSV = Path("index/wsc_with_disease.csv")

# ===== 1) 读 wsc_path_indices，并提取 session_id =====
# 示例：
# /.../wsc-visit1-10119-nsrr.npz -> session_id: wsc-visit1-10119-nsrr
df = pd.read_csv(PATH_CSV)
df["session_id"] = df["path"].str.extract(r"([^/]+)\.npz$", expand=False)

# ===== 2) 读取新疾病标签，只保留 dataset == 'wsc'，标准化 patient_id =====
lab = pd.read_csv(LABEL_CSV, low_memory=False)
lab = lab[lab["dataset"].astype(str).str.lower().eq("wsc")].copy()

def normalize_pid(x: str) -> str:
    s = str(x).strip()
    s = s.lstrip("/")
    s = s.split("?")[0].split("#")[0]
    s = s.split(".")[0]   # 去掉可能的扩展名（如 .npz）
    return s

lab["patient_id"] = lab["patient_id"].apply(normalize_pid)

# 疾病列全集（除 dataset、patient_id 外全部保留）
disease_cols = [c for c in lab.columns if c not in ("dataset", "patient_id")]

# 将疾病列统一为可空整型 Int64（0/1/NA）
for c in disease_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce").round().astype("Int64")

lab = lab[["patient_id"] + disease_cols].drop_duplicates("patient_id")

# ===== 3) 合并（左连接）：session_id <-> patient_id =====
merged = df.merge(lab, left_on="session_id", right_on="patient_id", how="left")

# 安全删除右表 ID（存在才删，避免 KeyError）
if "patient_id" in merged.columns:
    merged = merged.drop(columns=["patient_id"])

# （可选）把疾病列放到表末尾
base_cols = [c for c in merged.columns if c not in disease_cols]
merged = merged[base_cols + disease_cols]

# ===== 4) 保存 =====
merged.to_csv(OUT_CSV, index=False)

# ===== 5) 简要统计 =====
n_total = len(df)
n_hit = merged[disease_cols].notna().any(axis=1).sum()
print(f"[OK] 已保存：{OUT_CSV}")
print(f"[INFO] 总样本数：{n_total}；至少匹配到一个疾病标签的样本数：{n_hit}")

unmatched = merged[merged[disease_cols].isna().all(axis=1)]
if len(unmatched) > 0:
    print(f"[WARN] 有 {len(unmatched)} 个样本未匹配到疾病标签，示例：")
    show_cols = ["path", "session_id"]
    print(unmatched[show_cols].head(10).to_string(index=False))

[OK] 已保存：index/wsc_with_disease.csv
[INFO] 总样本数：2519；至少匹配到一个疾病标签的样本数：2519


## apples

In [2]:
import pandas as pd
import re
from pathlib import Path

# --- 读取 CSV ---
df_idx = pd.read_csv("index/apples_path_indices.csv")   # path, patient_id(无用), split, duration
df_meta = pd.read_csv("index/apples_metadata.csv")      # age,bmi,ethnicity,race,sexM,dataset,patient_id,session_id

# --- 1) 从 path 提取 session_id：/..../apples-130001.npz -> apples-130001
df_idx["session_id"] = df_idx["path"].str.extract(r"([^/]+)\.npz$", expand=False)

# 说明：apples_path_indices.csv 里的 patient_id 实际是文件名（含 .npz），不可靠，直接丢弃
df_idx = df_idx.drop(columns=[c for c in ["patient_id"] if c in df_idx.columns])

# --- 2) 合并：按 session_id 对齐（metadata 才有真正的 patient_id，如 APL0001）---
merged = df_idx.merge(df_meta, on="session_id", how="left")

# --- 3) 列名映射（与 mros 模板保持一致）---
columns_mapping = {
    "patient_id": "subject_id",
    "session_id": "session",
    "sexM": "sex",
    "bmi": "bmi",
    "age": "age",
    "ethnicity": "ethnicity",
    "race": "Race",
    "dataset": "Dataset",
    "path": "path",
    "duration": "duration",
    "split": "split",
}

merged = merged.rename(columns=columns_mapping)

# （可选）列顺序，和模板对齐
wanted_order = [
    "subject_id", "session", "sex", "bmi", "age", "ethnicity", "Race",
    "Dataset", "path", "duration", "split"
]
# 仅重排存在的列
cols = [c for c in wanted_order if c in merged.columns] + \
       [c for c in merged.columns if c not in wanted_order]
merged = merged[cols]

# --- 4) 保存结果 ---
out_path = Path("index/apples_psg_pretrain.csv")
merged.to_csv(out_path, index=False, encoding="utf-8")
print(f"[OK] 保存至 {out_path}")


[OK] 保存至 index/apples_psg_pretrain.csv


In [3]:
import pandas as pd
from pathlib import Path

# ===== 路径 =====
BASE_CSV = Path("index/apples_psg_pretrain.csv")     # 你刚得到的 apples 基础索引
LABEL_CSV = Path("index/disease_index_full.csv")     # 跨数据集疾病标签
OUT_CSV = Path("index/apples_with_disease.csv")      # 输出

# ===== 1) 读取 apples 基础表 =====
base = pd.read_csv(BASE_CSV)

# ===== 2) 读取疾病标签，只保留 dataset == 'apples'，并标准化 patient_id =====
lab = pd.read_csv(LABEL_CSV, low_memory=False)

lab = lab[lab["dataset"].astype(str).str.lower().eq("apples")].copy()

def normalize_pid(x: str) -> str:
    s = str(x).strip()
    s = s.lstrip("/")                # 去掉可能的前导斜杠
    s = s.split("?")[0].split("#")[0]
    s = s.split(".")[0]              # 去掉扩展名（如 .npz）
    return s

lab["patient_id"] = lab["patient_id"].apply(normalize_pid)

# 疾病列全集（除 dataset、patient_id 外全部保留）
disease_cols = [c for c in lab.columns if c not in ("dataset", "patient_id")]

# 将疾病列统一为可空整型 Int64（0/1/NA）
for c in disease_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce").round().astype("Int64")

# 去重：以 patient_id 为主键
lab = lab[["patient_id"] + disease_cols].drop_duplicates("patient_id")

# ===== 3) 合并（左连接）：session <-> patient_id =====
merged = base.merge(lab, left_on="session", right_on="patient_id", how="left")

# 安全删除右表 ID（存在才删）
if "patient_id" in merged.columns:
    merged = merged.drop(columns=["patient_id"])

# （可选）把疾病列放到表末尾
base_cols = [c for c in merged.columns if c not in disease_cols]
merged = merged[base_cols + disease_cols]

# ===== 4) 保存 =====
merged.to_csv(OUT_CSV, index=False)
print(f"[OK] 已保存：{OUT_CSV}")

# ===== 5) 简要统计 =====
n_total = len(base)
n_hit = merged[disease_cols].notna().any(axis=1).sum()
print(f"[INFO] 总样本数：{n_total}；至少匹配到一个疾病标签的样本数：{n_hit}")

unmatched = merged[merged[disease_cols].isna().all(axis=1)]
if len(unmatched) > 0:
    print(f"[WARN] 有 {len(unmatched)} 个样本未匹配到疾病标签，示例：")
    show_cols = ["subject_id", "session", "path"]
    show_cols = [c for c in show_cols if c in unmatched.columns]
    print(unmatched[show_cols].head(10).to_string(index=False))


[OK] 已保存：index/apples_with_disease.csv
[INFO] 总样本数：1082；至少匹配到一个疾病标签的样本数：1082


### 合并数据集

In [ ]:
import pandas as pd
import glob

# 1. 定义要读取的文件列表
csv_files = [
    # "index/hsp_psg_pretrain.csv",
    # "index/mesa_psg_pretrain.csv",
    # "index/mros_psg_pretrain.csv",
    "index/shhs_psg_pretrain.csv"
]

# 2. 读取并只保留指定列
dfs = []
for file in csv_files:
    df = pd.read_csv(file, usecols=["path", "age", "sex", "split", "duration"])
    dfs.append(df)

# 3. 合并为一个 DataFrame
merged_df = pd.concat(dfs, ignore_index=True)

# 4. 保存到新文件
target_name = "merged_psg_finetune.csv"
merged_df.to_csv("merged_psg_finetune.csv", index=False)

print("合并完成，输出文件: merged_psg_finetune.csv")


合并完成，输出文件: merged_psg_pretrain.csv


#### 检查

In [27]:
import numpy as np

def inspect_npz(path: str, show_data: bool = False, max_elements: int = 10) -> None:
    """
    查看 npz 文件中的所有数据。
    
    Args:
        path (str): npz 文件路径。
        show_data (bool): 是否显示部分数据值（默认 False）。
        max_elements (int): 显示数据值的最大元素数量。
    """
    with np.load(path) as data:
        print(f"文件包含 {len(data.files)} 个数组：")
        for key in data.files:
            arr = data[key]
            print(f"--- {key} ---")
            print(f"  形状：{arr.shape}, 数据类型：{arr.dtype}")
            if show_data:
                # 展示数组的前几个元素
                flat = arr.flatten()
                print(f"  数据示例：{flat[:max_elements]}")
            print()
        print(data['stage5'].shape)
 
    return data

data = inspect_npz("/DATA/disk1/ywx/wuji/hsp_8_channels/sub-S0001117014298/ses-1.npz", show_data=False)

文件包含 11 个数组：
--- heartbeat ---
  形状：(106340,), 数据类型：float32

--- breath ---
  形状：(106340,), 数据类型：float32

--- eeg_original ---
  形状：(3402880,), 数据类型：float32

--- ecg_original ---
  形状：(1701440,), 数据类型：float32

--- emg_original ---
  形状：(3402880,), 数据类型：float32

--- eog_original ---
  形状：(3402880,), 数据类型：float32

--- resp_original ---
  形状：(106340,), 数据类型：float32

--- body_movement ---
  形状：(106340,), 数据类型：bool

--- spo2 ---
  形状：(106340,), 数据类型：float32

--- stage5 ---
  形状：(886,), 数据类型：int64

--- duration ---
  形状：(), 数据类型：float64

(886,)


In [22]:
data.files

['heartbeat',
 'breath',
 'eeg_original',
 'ecg_original',
 'emg_original',
 'eog_original',
 'resp_original',
 'body_movement',
 'spo2',
 'stage5',
 'duration']

In [23]:
data['stage5'].shape

AttributeError: 'NoneType' object has no attribute 'open'

### 数据覆盖率

In [2]:
import pandas as pd

# 读取 CSV 文件
csv_path = "/mnt/wuji/ywx/psg-pretrain/index/hsp_basic_info.csv"  # ← 替换成你的实际路径
df = pd.read_csv(csv_path)

# 要检查的列（按你提供的顺序列出）
columns_to_check = [
    "SiteID", "BDSPPatientID", "HashFolderName", "ShiftedCreationTime",
    "PreSleepQuestionnaire", "HasAnnotations", "HasStaging", "AgeAtVisit",
    "ShiftedDateOfDeath", "SexDSC", "EthnicGroupDSC", "MaritalStatusDSC",
    "LanguageDSC", "BDSPLastModifiedDTS", "visit"
]

# 计算每列的非空覆盖率
coverage = {}
total_rows = len(df)

for col in columns_to_check:
    if col not in df.columns:
        print(f"[Warning] 列 {col} 不存在于文件中，跳过")
        continue
    non_null_count = df[col].notnull().sum()
    coverage[col] = non_null_count / total_rows

# 输出覆盖率
print("\n列覆盖率（非空占比）：")
for col, rate in coverage.items():
    print(f"{col:25s} : {rate:.2%}")



列覆盖率（非空占比）：
SiteID                    : 100.00%
BDSPPatientID             : 100.00%
HashFolderName            : 100.00%
ShiftedCreationTime       : 100.00%
PreSleepQuestionnaire     : 100.00%
HasAnnotations            : 100.00%
HasStaging                : 100.00%
AgeAtVisit                : 100.00%
ShiftedDateOfDeath        : 5.91%
SexDSC                    : 100.00%
EthnicGroupDSC            : 99.98%
MaritalStatusDSC          : 99.96%
LanguageDSC               : 99.95%
BDSPLastModifiedDTS       : 100.00%
visit                     : 100.00%
